In [ ]:
!pwd
%cd ..

In [ ]:

import os

# This forces OpenMP to use 1 single thread, which is needed to
# prevent contention between multiple process.
os.environ['OMP_NUM_THREADS'] = '1'
# Tell numpy to only use one core.
os.environ['MKL_NUM_THREADS'] = '1'


import multiprocessing as mp
import sys
from absl import flags

import numpy as np
import torch
from torch.optim.lr_scheduler import MultiStepLR

FLAGS = flags.FLAGS

flags.DEFINE_integer('board_size', 5, 'Board size for Go.')
flags.DEFINE_float('komi', 7.5, 'Komi rule for Go.')
flags.DEFINE_integer(
    'num_stack',
    8,
    'Stack N previous states, the state is an image of N x 2 + 1 binary planes.',
)
flags.DEFINE_integer('num_res_blocks', 6, 'Number of residual blocks in the neural network.')
flags.DEFINE_integer('num_filters_resnet', 123, 'Number of filters for the conv2d layers in the neural network.')
flags.DEFINE_integer(
    'num_fc_units',
    128,
    'Number of hidden units in the linear layer of the neural network.',
)

flags.DEFINE_integer(
    'num_simulations',
    200,
    'Number of simulations per MCTS search, this applies to both self-play and evaluation processes.',
)

flags.DEFINE_integer(
    'num_parallel',
    8,
    'Number of leaves to collect before using the neural network to evaluate the positions during MCTS search,'
    '1 means no parallel search.',
)
flags.DEFINE_float(
    'c_puct_base',
    19652,
    'Exploration constants balancing priors vs. search values. Original paper use 19652',
)
flags.DEFINE_float(
    'c_puct_init',
    1.25,
    'Exploration constants balancing priors vs. search values. Original paper use 1.25',
)

flags.DEFINE_float(
    'default_rating',
    1500,
    'Default elo rating, change to the rating (for black) from last checkpoint when resume training.',
)
flags.DEFINE_string(
    'logs_dir',
    './logs/go/9x9/alphago_series',
    'Path to save statistics for self-play, training, and evaluation.',
)
flags.DEFINE_string('log_level', 'INFO', '')
flags.DEFINE_integer('seed', 1, 'Seed the runtime.')
# Initialize flags
FLAGS(sys.argv, known_only = True)

os.environ['BOARD_SIZE'] = str(FLAGS.board_size)

In [ ]:
from alpha_zero.envs.go import GoEnv
from alpha_zero.core.pipeline import (
    set_seed,
)
from alpha_zero.core.multi_game import run_tournament
from alpha_zero.core.quantum_net import QuantumAlphaZeroNet, SoftHardSearchAlphaZeroNet
from alpha_zero.core.network import AlphaZeroNet
from alpha_zero.utils.util import extract_args_from_flags_dict, create_logger

In [ ]:
def env_builder():
        return GoEnv(komi=FLAGS.komi, num_stack=FLAGS.num_stack)
eval_env = env_builder()

input_shape = eval_env.observation_space.shape
num_actions = eval_env.action_space.n

config = {
        'name': 'hybrid3',
        'num_filters': 105,
        'max_depth': 2,
        'branching_width': 3,
        'beam_width': 3,
        'num_fc_units':128,
        'num_search':3,
    }
# Replace SoftHardSearchAlphaZeroNet with QuantumAlphaZeroNet for softsearch version
hybridsearch_agent = SoftHardSearchAlphaZeroNet(
        input_shape,
        num_actions,
        config['num_filters'],
        config['max_depth'],
        config['branching_width'],
        config['beam_width'],
        config['num_fc_units'],
        config['num_search'],
    )
        
hybridsearch_chkpts= ['./checkpoints_RL/HybridSearch/training_steps_30720.ckpt',
                './checkpoints_RL/HybridSearch/training_steps_50688.ckpt',
                './checkpoints_RL/HybridSearch/training_steps_70144.ckpt',
                './checkpoints_RL/HybridSearch/training_steps_90112.ckpt',
                './checkpoints_RL/HybridSearch/training_steps_107008.ckpt']



In [ ]:
agents = {
    "HybridSearch_30k": {
        "network": hybridsearch_agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": hybridsearch_chkpts[0],
        "wins": 0,
        "lost": 0
    },
    "HybridSearch_50k": {
        "network": hybridsearch_agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": hybridsearch_chkpts[1],
        "wins": 0,
        "lost": 0
    },
    "HybridSearch_70k": {
        "network": hybridsearch_agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": hybridsearch_chkpts[2],
        "wins": 0,
        "lost": 0
    },
    "HybridSearch_90k": {
        "network": hybridsearch_agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": hybridsearch_chkpts[3],
        "wins": 0,
        "lost": 0
    },
    "HybridSearch_107k": {
        "network": hybridsearch_agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": hybridsearch_chkpts[4],
        "wins": 0,
        "lost": 0
    }
}

In [ ]:
len(agents)

In [ ]:
set_seed(FLAGS.seed)

logger = create_logger(FLAGS.log_level)

logger.info(extract_args_from_flags_dict(FLAGS.flag_values_dict()))


In [ ]:
if torch.cuda.is_available():
    learner_device = torch.device('cuda')

run_tournament(
    seed = FLAGS.seed,
    agents = agents,
    env = eval_env,
    device = learner_device,
    num_games = 1000*len(agents),
    num_simulations = FLAGS.num_simulations,
    num_parallel = FLAGS.num_parallel,
    c_puct_base = FLAGS.c_puct_base,
    c_puct_init = FLAGS.c_puct_init,
    default_rating = FLAGS.default_rating,
    log_level = FLAGS.log_level,
    logs_dir = FLAGS.logs_dir,

)